In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.pyplot as plt
import seaborn as sns

import json
import geopandas as gpd

import os

In [208]:
mapping_kecamatan = {
    "ASEMROWO": "ASEM ROWO",
    "DUKUHPAKIS": "DUKUH PAKIS",
    "GUNUNGANYAR": "GUNUNG ANYAR",
    "KARANGPILANG": "KARANG PILANG",
    "LAKARSANTRI": "LAKARSANTRI",
    "MULYOREJO": "MULYOREJO",
    "SAMBIKEREP": "SAMBIKEREP",
    "SUKOLILO": "SUKOLILO",
    "SUKOMANUNGGAL": "SUKOMANUNGGAL",
    "TENGGILISMEJOYO": "TENGGILIS MEJOYO",
    "WIYUNG": "WIYUNG",

    # Nama yang memang sudah sama
    "BENOWO": "BENOWO",
    "PAKAL": "PAKAL",
    "BUBUTAN": "BUBUTAN",
    "BULAK": "BULAK",
    "GAYUNGAN": "GAYUNGAN",
    "GENTENG": "GENTENG",
    "GUBENG": "GUBENG",
    "JAMBANGAN": "JAMBANGAN",
    "KENJERAN": "KENJERAN",
    "KREMBANGAN": "KREMBANGAN",
    "PABEAN CANTIAN": "PABEAN CANTIAN",
    "RUNGKUT": "RUNGKUT",
    "SAWAHAN": "SAWAHAN",
    "SEMAMPIR": "SEMAMPIR",
    "SIMOKERTO": "SIMOKERTO",
    "TAMBAKSARI": "TAMBAKSARI",
    "TANDES": "TANDES",
    "TEGALSARI": "TEGALSARI",
    "WONOCOLO": "WONOCOLO",
    "WONOKROMO": "WONOKROMO"
}

def normalisasi_kecamatan(df, kolom="Kecamatan"):
    df[kolom] = (
        df[kolom]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)   # rapikan spasi
    )

    mapping = {
        "ASEMROWO": "ASEM ROWO",
        "DUKUHPAKIS": "DUKUH PAKIS",
        "GUNUNGANYAR": "GUNUNG ANYAR",
        "KARANGPILANG": "KARANG PILANG",
        "TENGGILISMEJOYO": "TENGGILIS MEJOYO",
    }

    df[kolom] = df[kolom].replace(mapping)
    return df

In [264]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km

    lat1, lon1 = radians(lat1), radians(lon1)
    lat2, lon2 = radians(lat2), radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

def idw(df, target_kecamatan, value_col, power=2):

    # lokasi target
    target = df[df["Kecamatan"] == target_kecamatan].iloc[0]

    # hanya data yang diketahui
    known = df[df[value_col].notna()].copy()

    # hitung jarak
    known["distance"] = known.apply(
        lambda row: haversine(
            target["lat"],
            target["long"],
            row["lat"],
            row["long"]
        ),
        axis=1
    )

    # hindari pembagian nol
    known = known[known["distance"] > 0]

    # bobot IDW
    known["weight"] = 1 / (known["distance"] ** power)

    # prediksi
    pred = (
        (known["weight"] * known[value_col]).sum()
        / known["weight"].sum()
    )

    return pred
koordinat = pd.read_excel("D:\KULIAH\Project\womanguard-index-surabaya\Koordinat.xlsx")
koordinat.head()

,No,Kecamatan,lat,long
0,1,Karang Pilang,-7.341345,112.695990
1,2,Wonocolo,-7.319820,112.742027
2,3,Rungkut,-7.319019,112.804593
3,4,Wonokromo,-7.303957,112.736011
4,5,Tegalsari,-7.279848,112.736069


# X1

In [290]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X1 Persentase Penduduk Miskin"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x1_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [291]:
jumlah_penduduk = pd.read_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X1 Persentase Penduduk Miskin\jumlah_penduduk_surabaya_per_kecamatan_2023_2025.csv")

In [292]:
#Imputasi Inverse Distance Weighting (IDW)
data = x1_2024.merge(koordinat, on="Kecamatan")
prediksi = idw(
    data,
    target_kecamatan="Tandes",
    value_col="Jumlah (Jiwa)",
    power=2
)

print(prediksi)

2165.438196269988


In [293]:
x1_2024.loc[
    x1_2024["Kecamatan"]=="Tandes",
    "Jumlah (Jiwa)"
] = round(prediksi)

In [321]:
gabungan = []

# Normalisasi data jumlah penduduk cukup sekali
jumlah_penduduk = normalisasi_kecamatan(jumlah_penduduk)

for tahun in range(2023, 2026):
    df = globals()[f"x1_{tahun}"].copy()

    # Format data x1 berbeda
    if tahun == 2023:
        temp = pd.DataFrame({
            "Kecamatan": df.iloc[:, 0],
            "Tahun": tahun,
            "Keluarga Miskin": pd.to_numeric(df.iloc[:, 1], errors="coerce")
        })
    else:
        temp = pd.DataFrame({
            "Kecamatan": df["Kecamatan"],
            "Tahun": tahun,
            "Keluarga Miskin": pd.to_numeric(df["Jumlah (Jiwa)"], errors="coerce")
        })

    # Normalisasi nama kecamatan
    temp = normalisasi_kecamatan(temp)

    # Hapus baris total kota
    temp = temp[
        ~temp["Kecamatan"].isin(["JUMLAH", "KOTA SURABAYA"])
    ].reset_index(drop=True)

    # Merge berdasarkan Kecamatan dan Tahun
    temp = temp.merge(
        jumlah_penduduk,
        on=["Kecamatan", "Tahun"],
        how="left"
    )

    # Hitung rasio
    temp["Rasio Kemiskinan"] = (
        temp["Keluarga Miskin"] /
        temp["Jumlah Penduduk"]
    ) * 100

    gabungan.append(temp)

# Gabungkan seluruh tahun
x1_final = pd.concat(gabungan, ignore_index=True)

# Export
x1_final.to_csv(
    r"D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X1 Persentase Penduduk Miskin\X1_Rasio_Keluarga_Miskin.csv",
    index=False,
    encoding="utf-8-sig"
)

x1_final.head()

,Kecamatan,Tahun,Keluarga Miskin,Jumlah Penduduk,Rasio Kemiskinan
0,TAMBAKSARI,2023,19654,226995,8.658340
1,WONOKROMO,2023,11973,154995,7.724765
2,SUKOMANUNGGAL,2023,3410,104786,3.254252
3,SEMAMPIR,2023,15171,182371,8.318757
4,GUBENG,2023,9613,133804,7.184389


# X2

In [317]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X2 Persentase Kepala Keluarga Perempuan"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x2_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [310]:
for tahun in range(2023, 2026):
    print(f"===== {tahun} =====")
    print(globals()[f"x2_{tahun}"].columns.tolist())

===== 2023 =====
['No', 'Kecamatan', 'Laki-laki', 'Perempuan', 'Jumlah']
===== 2024 =====
['No', 'Kecamatan', 'Laki-laki', 'Perempuan', 'Jumlah']
===== 2025 =====
['Kecamtan', 'Kepala KeluargaPerempuan', 'Kepala keluarga laki laki']


In [318]:
gabungan = []

for tahun in range(2023, 2026):
    df = globals()[f"x2_{tahun}"].copy()

    if tahun != 2025:
        temp = pd.DataFrame({
            "Kecamatan": df.iloc[:, 1],
            "Tahun": tahun,
            "Rasio Kepala Keluarga Perempuan": df.iloc[:, 3]/(df.iloc[:,2] + df.iloc[:,3])*100
        })
    else:
        temp = pd.DataFrame({
            "Kecamatan": df.iloc[:, 0],
            "Tahun": tahun,
            "Rasio Kepala Keluarga Perempuan": df.iloc[:, 1]/(df.iloc[:,1] + df.iloc[:,2])*100
        })

    # Tambahkan ke list untuk semua tahun
    gabungan.append(temp)

# Gabungkan semua tahun
x2_final = pd.concat(gabungan, ignore_index=True)

# Export ke CSV
x2_final.to_csv(
    r"D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X2 Persentase Kepala Keluarga Perempuan\X2_Rasio Kepala Keluarga Perempuan.csv",
    index=False,
    encoding="utf-8-sig"
)
x2_final.head()

,Kecamatan,Tahun,Rasio Kepala Keluarga Perempuan
0,Karang Pilang,2023,22.031423
1,Wonocolo,2023,23.825202
2,Rungkut,2023,21.401099
3,Wonokromo,2023,27.068295
4,Tegalsari,2023,28.310238


# X4

In [138]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4 Sex Ratio"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x4_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [139]:
for tahun in range(2023, 2026):
    print(f"===== {tahun} =====")
    print(globals()[f"x4_{tahun}"].columns.tolist())

===== 2023 =====
['No', 'Kecamatan', 'Rasio Jenis Kelamin']
===== 2024 =====
['No', 'Kecamatan', 'Rasio Jenis Kelamin']
===== 2025 =====
['No', 'Kecamatan', 'Rasio Jenis Kelamin']


In [64]:
gabungan = []

for tahun in range(2023, 2026):
    df = globals()[f"x4_{tahun}"].copy()

    # Ambil hanya kolom yang diperlukan
    temp = pd.DataFrame({
        "Kecamatan": df.iloc[:, 1],  # kolom pertama
        "Tahun": tahun,
        "Sex Ratio": df.iloc[:,2]
    })

    gabungan.append(temp)

# Gabungkan semua tahun
x4_final = pd.concat(gabungan, ignore_index=True)

# Export ke CSV
x4_final.to_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X4_Sex_Ratio (%).csv",
                index=False,
                encoding="utf-8-sig")

print(x4_final.head())

          Kecamatan  Tahun  Sex Ratio
0      Karangpilang   2023      101.0
1         Jambangan   2023       98.0
2          Gayungan   2023       96.0
3          Wonocolo   2023       98.0
4  Tenggilis Mejoyo   2023       97.0


# X5

In [131]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X5 Persentase kepala keluarga perempuan yang bekerja"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x5_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [132]:
gabungan = []

for tahun in range(2023, 2026):
    df = globals()[f"x5_{tahun}"].copy()

    # Hapus kolom kosong jika ada
    df = df.drop(columns=df.columns[-1])

    # ==========================
    # Gabungkan header
    # ==========================
    header1 = df.iloc[3].fillna("").astype(str)
    header2 = df.iloc[4].fillna("").astype(str)

    kolom = []
    for h1, h2 in zip(header1, header2):
        h1 = h1.strip()
        h2 = h2.strip()

        if h1 == "":
            kolom.append(h2)
        elif h2 == "":
            kolom.append(h1)
        else:
            kolom.append(f"{h1}_{h2}")

    df.columns = kolom

    # Hapus baris header
    df = df.iloc[5:].reset_index(drop=True)

    # ==========================
    # Perbaiki kolom PR
    # ==========================
    kolom_baru = []
    nama_terakhir = None

    for col in df.columns:
        if col.endswith("_LK"):
            nama_terakhir = col[:-3]
            kolom_baru.append(col)

        elif col == "PR":
            kolom_baru.append(f"{nama_terakhir}_PR")

        else:
            kolom_baru.append(col)

    df.columns = (
        pd.Index(kolom_baru)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    # ==========================
    # Ambil hanya data kecamatan
    # ==========================
    df = df[df.iloc[:, 0] == "KECAMATAN"].reset_index(drop=True)

    # Ubah seluruh kolom angka menjadi numerik
    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Hitung bekerja & tidak bekerja
    tidak_bekerja = df.iloc[:, 2:10:2].sum(axis=1)
    bekerja = df.iloc[:, 10:-1:2].sum(axis=1)

    temp = pd.DataFrame({
        "Kecamatan": df["NAMA_KELURAHAN"],
        "Tahun": tahun,
        "Tidak Bekerja": tidak_bekerja,
        "Bekerja": bekerja,
    })

    temp["Persentase KK Perempuan Bekerja"] = (
        temp["Bekerja"] /
        (temp["Bekerja"] + temp["Tidak Bekerja"])
    ) * 100

    gabungan.append(temp)

# Gabungkan seluruh tahun
x5_final = pd.concat(gabungan, ignore_index=True)

# Export
x5_final.to_csv(
    r"D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X5 Persentase kepala keluarga perempuan yang bekerja\X5_Persentase_KK_Perempuan_Bekerja (%).csv",
    index=False,
    encoding="utf-8-sig"
)

print(x5_final.head())

       Kecamatan  Tahun  Tidak Bekerja  Bekerja  \
0  KARANG PILANG   2023           1884    17867   
1       WONOCOLO   2023           1928    18432   
2        RUNGKUT   2023           2528    28943   
3      WONOKROMO   2023           3663    35742   
4      TEGALSARI   2023           1682    23099   

   Persentase KK Perempuan Bekerja  
0                        90.461242  
1                        90.530452  
2                        91.967208  
3                        90.704225  
4                        93.212542  


# X6

In [2]:
folder = "D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X6 Jumlah Lulus"

for file in os.listdir(folder):
    if file.endswith(".xlsx"):
        tahun = file.split()[-1].replace(".xlsx", "")
        globals()[f"x6_{tahun}"] = pd.read_excel(os.path.join(folder, file))

In [3]:
def olah_x6(df):
    df = df[3:].copy()
    df = df.drop(columns=['Unnamed: 21'])
    df = df[df.iloc[:, 0] != 'Kota Surabaya']
    df['ga_lulus'] = 0
    df['lulus'] = 0
    for i in range(1,21):
        data = df.iloc[:,i]
        if i%2==0 and i<=10:
            df['ga_lulus'] += data
        else:
            df['lulus'] += data
    
    df['Jumlah_perempuan_lulus_sma'] = df['lulus'] - df['ga_lulus']
    return df


In [5]:
for tahun in range(2023, 2026):
    globals()[f"x6_{tahun}"] = olah_x6(globals()[f"x6_{tahun}"])

In [7]:
gabungan = []

for tahun in range(2023, 2026):
    df = globals()[f"x6_{tahun}"].copy()

    # Ambil hanya kolom yang diperlukan
    temp = pd.DataFrame({
        "Kecamatan": df.iloc[:, 0],  # kolom pertama
        "Tahun": tahun,
        "Jumlah_perempuan_lulus_sma": df["Jumlah_perempuan_lulus_sma"]
    })

    gabungan.append(temp)

# Gabungkan semua tahun
x6_final = pd.concat(gabungan, ignore_index=True)

# Export ke CSV
x6_final.to_csv("D:\KULIAH\Project\womanguard-index-surabaya\Data terbaru dip\X6 Jumlah Lulus\X6_Jumlah_Perempuan_Lulus_SMA_2023_2025.csv",
                index=False,
                encoding="utf-8-sig")

print(x6_final.head())

          Kecamatan  Tahun Jumlah_perempuan_lulus_sma
0      Karangpilang   2023                      11565
1         Jambangan   2023                      10983
2          Gayungan   2023                       9607
3          Wonocolo   2023                      13313
4  Tenggilis Mejoyo   2023                      10593
